In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import densenet201

In [ ]:
class DenseLayer(nn.Module):
    """A single dense layer module as described in the DenseNet architecture.

    This layer implements the bottleneck design, where a 1x1 convolution reduces
    the number of feature maps before a 3x3 convolution is applied. The output
    feature maps are then concatenated with the input feature maps.

    Args:
        in_channels (int): The number of **input channels**.
        growth_rate (int): The number of feature maps to produce (**k** in the paper).
        bn_size (int): The multiplicative factor for the number of bottleneck channels.
    """
    def __init__(self, in_channels, growth_rate=32, bn_size=4):
        super(DenseLayer, self).__init__()

        # In channels: 3
        # out channels: 32 * 4 = 128

        # Bottleneck layer: 1x1 convolution for dimensionality reduction.
        self.dimension_reduction = nn.Sequential(
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                in_channels, bn_size * growth_rate, kernel_size=1, stride=1, bias=False
            ),
        )


        # 128 -> 32
        # Feature extraction layer: 3x3 convolution to generate new features.
        self.feature_extraction = nn.Sequential(
            nn.BatchNorm2d(bn_size * growth_rate),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                bn_size * growth_rate,
                growth_rate,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
        )

    def forward(self, x):
        """Defines the forward pass of the dense layer.

        Args:
            x (torch.Tensor): The input tensor.

        Returns:
            torch.Tensor: The output tensor after concatenating with the input.
        """
        # Pass the input through the bottleneck and feature extraction layers.

        # Example: increase to 128 channels, then reduce to 32 channels.
        new_features = self.dimension_reduction(x)
        new_features = self.feature_extraction(new_features)
        
        # Concatenate the new feature maps with the original input feature maps.
        # concatenate X with new 32 channels
        concatenated_features = torch.cat((x, new_features), 1)

        return concatenated_features

In [10]:
densenet_201 = DenseNet201ReviewKD(num_classes=10, pretrained=False)

In [11]:
# pass dummy input to get the feature maps
dummy_input = torch.randn(1, 3, 224, 224)
output, feats = densenet_201(dummy_input)


In [12]:
# check shapes of feature maps
for i, feat in enumerate(feats["preact_feats"]):
    print(f"Feature map {i} shape: {feat.shape}")
    

Feature map 0 shape: torch.Size([1, 1920, 1, 1])
Feature map 1 shape: torch.Size([1, 1920, 7, 7])
Feature map 2 shape: torch.Size([1, 1792, 14, 14])
Feature map 3 shape: torch.Size([1, 512, 28, 28])
Feature map 4 shape: torch.Size([1, 256, 56, 56])


In [16]:
from torchvision.models import mobilenet_v2

# pass dummy input to get the feature maps
dummy_input = torch.randn(1, 3, 224, 224)
mobilenet_v2_model = mobilenet_v2(pretrained=False)
output = mobilenet_v2_model(dummy_input)

In [23]:
mobilenet_v2_model.features

Sequential(
  (0): Conv2dNormActivation(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU6(inplace=True)
  )
  (1): InvertedResidual(
    (conv): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
      )
      (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
  )
  (2): InvertedResidual(
    (conv): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (

In [25]:
# print output shape of each feature map in mobilenet_v2_model.features
for i, layer in enumerate(mobilenet_v2_model.features):
    # pass dummy input through the layer
    output = layer(dummy_input)
    print(f"Feature map {i} shape: {output.shape}")
    # update dummy_input for the next layer
    dummy_input = output

Feature map 0 shape: torch.Size([1, 32, 112, 112])
Feature map 1 shape: torch.Size([1, 16, 112, 112])
Feature map 2 shape: torch.Size([1, 24, 56, 56])
Feature map 3 shape: torch.Size([1, 24, 56, 56])
Feature map 4 shape: torch.Size([1, 32, 28, 28])
Feature map 5 shape: torch.Size([1, 32, 28, 28])
Feature map 6 shape: torch.Size([1, 32, 28, 28])
Feature map 7 shape: torch.Size([1, 64, 14, 14])
Feature map 8 shape: torch.Size([1, 64, 14, 14])
Feature map 9 shape: torch.Size([1, 64, 14, 14])
Feature map 10 shape: torch.Size([1, 64, 14, 14])
Feature map 11 shape: torch.Size([1, 96, 14, 14])
Feature map 12 shape: torch.Size([1, 96, 14, 14])
Feature map 13 shape: torch.Size([1, 96, 14, 14])
Feature map 14 shape: torch.Size([1, 160, 7, 7])
Feature map 15 shape: torch.Size([1, 160, 7, 7])
Feature map 16 shape: torch.Size([1, 160, 7, 7])
Feature map 17 shape: torch.Size([1, 320, 7, 7])
Feature map 18 shape: torch.Size([1, 1280, 7, 7])


In [26]:
mobilenet_v2_model

MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [46]:
import torch
import torchvision.models as models

# Initialize MobileNetV3-Small with 0.35 width multiplier
model_small = models.mobilenet_v3_small(width_mult=0.50, num_classes=10)


# Test with a dummy input (Batch size 1, 3 channels, 224x224 image)
dummy_input = torch.randn(1, 3, 224, 224)
output = model_small(dummy_input)

print("Output shape:", output.shape)  # torch.Size([1, 1000])


Output shape: torch.Size([1, 10])


In [47]:
# count param of model_small
total_params = sum(p.numel() for p in model_small.parameters())
print(f"Total parameters in MobileNetV3-Small (0.35 width multiplier): {total_params}")

Total parameters in MobileNetV3-Small (0.35 width multiplier): 409394


In [44]:
import torch
import torchvision.models as models

# 1. Load the ShuffleNet V2 0.5x model with pretrained ImageNet weights
weights = models.ShuffleNet_V2_X0_5_Weights.DEFAULT
shufflenet_model = models.shufflenet_v2_x0_5(weights=None, num_classes=10)

# 2. Set the model to evaluation mode
shufflenet_model.eval()

# 3. Create dummy input matching ImageNet dimensions (Batch size, Channels, Height, Width)
# ShuffleNet V2 expects 3-channel RGB images of size 224x224
dummy_input = torch.randn(1, 3, 224, 224)

# 4. Forward pass
with torch.no_grad():
    output = shufflenet_model(dummy_input)

print(f"Output tensor shape: {output.shape}")  # Expected: torch.Size([1, 1000])


Output tensor shape: torch.Size([1, 10])


In [45]:
# count param of shufflenet_model
total_params = sum(p.numel() for p in shufflenet_model.parameters())
print(f"Total parameters in ShuffleNet V2 (0.5x width multiplier): {total_params}")

Total parameters in ShuffleNet V2 (0.5x width multiplier): 352042
